Assignment 3


* Combine what you learnt about word vectors with other NLP techniques in spaCy
* It shall take user inputs and generate some simple outputs
* It can be framed as a little word game, a tool that compares two documents, a name generator based on similarity, alternative word suggestions based on word vectors, etc...
* Keep it small and it doesn't need to work perfectly yet. If something didn't work out as you expected, write it down in comments



In [20]:
import numpy as np

In [13]:
import spacy
!python -m spacy download en_core_web_lg
nlp = spacy.load("en_core_web_lg")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
# import the text 'Romeo and Juliet' from Google Drive
# the text is from https://www.gutenberg.org/ebooks/1513
from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/My Drive/Leuphana U/Semester-5/NLP/Assignment 3/RomeoJuliet.txt") as f:
    text = f.read()

Mounted at /content/drive


In [14]:
# create a list of vocabulary from the text by tokenization
doc = nlp(text)
vocab = [token.text for token in doc if token.is_alpha]

### The functions below are from the seminar

In [17]:
def nearest_neighbors(target, vocab, k=5):
    """
    Find the nearest neighbors of a target word within a vocab
    vocab needs to be a list of strings
    """
    query = nlp(target)[0]  # token
    scores = {}
    for w in vocab:
        token = nlp.vocab[w]
        if w == target: # skip the same word
            continue
        if token.has_vector:
            score = query.similarity(token)
            scores[w] = score
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return top

In [26]:
def analogy(a, b, c, top_n=1):
    """
    Solve analogies of the form: A : B :: C : D
    Returns the best matching word D.
    """
    # Get vector representations
    A = nlp.vocab[a].vector
    B = nlp.vocab[b].vector
    C = nlp.vocab[c].vector

    # Compute target vector
    target = B - A + C

    # Search the entire vocabulary for closest words
    similarities = []
    for w in vocab: # only check common_words
        word = nlp.vocab[w] # get lexeme
        if word.has_vector and word.is_alpha and not word.is_stop:
            sim = np.dot(target, word.vector) / (
                np.linalg.norm(target) * np.linalg.norm(word.vector)
            )
            similarities.append((word.text, float(sim)))

    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Return the top result(s), skipping the original input words
    results = [w for w, s in similarities if w not in {a, b, c}]
    return results[:top_n]

In [33]:
def nearest_neighbors_sentence(input, sentences, k=5):
    query = nlp(input)
    scores = {}
    for s in sentences:
        if not s.has_vector: # ignore sentences without vectors
            continue;
        score = query.similarity(s)
        scores[s.text.replace("\n"," ")] = score
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return top
    pass

### TEST

In [35]:
nearest_neighbors("rose", vocab, 10)

[('fell', 0.6220752596855164),
 ('roses', 0.5973121523857117),
 ('fallen', 0.5697540640830994),
 ('flower', 0.5493743419647217),
 ('Flower', 0.5493743419647217),
 ('rise', 0.5073449611663818),
 ('gold', 0.4993191957473755),
 ('silver', 0.4963153600692749),
 ('fall', 0.4871509373188019),
 ('flowers', 0.48484909534454346)]

In [36]:
analogy("cat", "bird", "dog", top_n=5)

['birds', 'Hunting', 'squirrel', 'flies', 'flies']

In [37]:
sentences = list(doc.sents)
nearest_neighbors_sentence("man with emotion", sentences, 5)

[('Unseemly woman in a seeming man, And ill-beseeming beast in seeming both! ',
  0.8250352144241333),
 ('Lady, such a man As all the world—why he’s a man of wax.   ',
  0.8029157519340515),
 ('Alas poor Romeo, he is already dead, stabbed with a white wench’s black eye; run through the ear with a love song, the very pin of his heart cleft with the blind bow-boy’s butt-shaft.',
  0.8022766709327698),
 ('And yet no man like he doth grieve my heart.   ', 0.8012517094612122),
 ('Love is a smoke made with the fume of sighs; Being purg’d, a fire sparkling in lovers’ eyes; Being vex’d, a sea nourish’d with lovers’ tears: What is it else?',
  0.7953416109085083)]

### Shakespeare's Sentence Suggestion Program

This program takes an input of a phrase from the user and suggests some relevant sentences from the text 'Romeo and Juliet'



In [38]:
input = str(input('please write a phrase or sentence: '))

nearest_neighbors_sentence(input, sentences, 5)

please write a phrase or sentence: red roses


[('Pink for flower.   ', 0.7769336104393005),
 ('Sweet flower, with flowers thy bridal bed I strew. O woe, thy canopy is dust and stones, Which with sweet water nightly I will dew, Or wanting that, with tears distill’d by moans. ',
  0.630736231803894),
 ('Lady, by yonder blessed moon I vow, That tips with silver all these fruit-tree tops,—   JULIET. ',
  0.6269209384918213),
 ('JULIET. O serpent heart, hid with a flowering face! ', 0.6117015480995178),
 ('Give me those flowers.', 0.5955024361610413)]